In [ ]:
from main.core_data.ds.td_dataset import TdSegmentedExperimentDataset

ds = TdSegmentedExperimentDataset(dataset_path=amigos_path, dataset_spec_file=spec_file, accessible_user_ids=ids)
ds[0]

In [ ]:
import tensordict
from torch.utils.data import DataLoader

max_duration = 120


def td_collate(batch):
    # TODO We need custom collate on label values (we only want first 5)
    # Take only first 3 minutes of each sample
    batch = [b.exclude("meta", ("assessment", "scales"), ("assessment", "labels"), )[:15] for b in batch]
    return tensordict.pad_sequence(batch, 0, return_mask="pad_mask")


dl = DataLoader(ds, shuffle=True, batch_size=32, collate_fn=td_collate)

In [ ]:
itdl = iter(dl)
next(itdl)

In [ ]:
from main.model.downstream.fusion_probe.data_utils import LinearProbeDataModule

deap_path = "/mnt/datasets/EEGAVI/DOWNSTREAM/interleaved-downstream-deap"
amigos_path = "/mnt/datasets/EEGAVI/DOWNSTREAM/interleaved-downstream"

dm = LinearProbeDataModule(seed=1, batch_size=32)
dm.add_dataset(amigos_path, 1, valid_fraction=0.1)
dm.add_dataset(deap_path, 1, test_fraction=1.0)

dm.setup("train")
train_dl = dm.train_dataloader()
test_dl = dm.test_dataloader()

In [ ]:
next(iter(train_dl))

In [ ]:
next(iter(test_dl))